# Google Play Store Apps: Story & Insights

This notebook focuses on the main insights from the Google Play Store dataset.
Instead of exploring all variables, the goal is to identify the most important patterns that can guide the design of a final interactive dashboard.

The main questions are:
1. Which app categories dominate the store?
2. Which categories are the most popular?
3. How do ratings relate to popularity?
4. What is the difference between free and paid apps?
5. Which variables should be emphasized in the final dashboard?

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

ModuleNotFoundError: No module named 'plotly'

In [ ]:
apps = pd.read_csv("../data/raw/googleplaystore.csv")
reviews = pd.read_csv("../data/raw/googleplaystore_user_reviews.csv")

In [ ]:
apps.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


## Relevant Features and Variables

For this analysis, the most relevant variables are the ones that help explain app popularity, user satisfaction, and pricing patterns.

The key columns selected are:

- **App**: app name, useful for identifying individual examples in plots
- **Category**: shows the app segment and helps compare groups
- **Rating**: reflects user satisfaction
- **Reviews**: indicates user engagement and popularity
- **Installs**: one of the strongest measures of popularity
- **Type**: distinguishes between free and paid apps
- **Price**: helps analyze monetization strategy
- **Content Rating**: useful for understanding audience targeting


These variables were selected because they are the most relevant for explaining app popularity, user satisfaction, and pricing patterns. Other variables were not included because they are less important for the main story of this analysis.

In [ ]:
relevant_cols = ["App", "Category", "Rating", "Reviews", "Installs", "Type", "Price", "Content Rating"]
apps[relevant_cols].head()

,App,Category,Rating,Reviews,Installs,Type,Price,Content Rating
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,"10,000+",Free,0,Everyone
1,Coloring book moana,ART_AND_DESIGN,3.9,967,"500,000+",Free,0,Everyone
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,"5,000,000+",Free,0,Everyone
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,"50,000,000+",Free,0,Teen
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,"100,000+",Free,0,Everyone


## Data Cleaning


In [ ]:
#Create a smaller dataframe

apps_story = apps[relevant_cols].copy()


In [ ]:
#Check missing values

apps_story.isnull().sum()

App                  0
Category             0
Rating            1474
Reviews              0
Installs             0
Type                 1
Price                0
Content Rating       1
dtype: int64

In [ ]:

#Remove impossible ratings

apps_story = apps_story[apps_story["Rating"] <= 5]


In [ ]:
#Convert Reviews to numeric

apps_story["Reviews"] = pd.to_numeric(apps_story["Reviews"], errors="coerce")

In [ ]:
#Clean Installs

apps_story["Installs"] = apps_story["Installs"].astype(str).str.replace(",", "", regex=False)
apps_story["Installs"] = apps_story["Installs"].str.replace("+", "", regex=False)
apps_story["Installs"] = pd.to_numeric(apps_story["Installs"], errors="coerce")

In [ ]:
#Clean Price

apps_story["Price"] = apps_story["Price"].astype(str).str.replace("$", "", regex=False)
apps_story["Price"] = pd.to_numeric(apps_story["Price"], errors="coerce")

In [ ]:
#Drop rows missing key values

apps_story = apps_story.dropna(subset=["Category", "Rating", "Reviews", "Installs", "Type", "Price"])

In [ ]:
apps_story.info()
apps_story.head()

<class 'pandas.core.frame.DataFrame'>
Index: 9366 entries, 0 to 10840
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             9366 non-null   object 
 1   Category        9366 non-null   object 
 2   Rating          9366 non-null   float64
 3   Reviews         9366 non-null   int64  
 4   Installs        9366 non-null   int64  
 5   Type            9366 non-null   object 
 6   Price           9366 non-null   float64
 7   Content Rating  9366 non-null   object 
dtypes: float64(2), int64(2), object(4)
memory usage: 658.5+ KB


,App,Category,Rating,Reviews,Installs,Type,Price,Content Rating
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,10000,Free,0.0,Everyone
1,Coloring book moana,ART_AND_DESIGN,3.9,967,500000,Free,0.0,Everyone
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,5000000,Free,0.0,Everyone
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,50000000,Free,0.0,Teen
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,100000,Free,0.0,Everyone


The dataset is now cleaned for the variables needed in the story.
The main preparation steps were:
- removing invalid rating values
- converting Reviews, Installs, and Price into numeric format
- dropping rows with missing values in the most important columns

This cleaned dataset will now be used for the focused visual analysis.


## Main Insights from the Data

After preparing the dataset, the next step is to focus on the most important patterns that help explain app market structure, popularity, ratings, and pricing.  
The following visualizations are selected to support the main story of the analysis.

In [ ]:
category_counts = apps_story["Category"].value_counts().reset_index()
category_counts.columns = ["Category", "Count"]

category_counts.head(10)

,Category,Count
0,FAMILY,1747
1,GAME,1097
2,TOOLS,734
3,PRODUCTIVITY,351
4,MEDICAL,350
5,COMMUNICATION,328
6,FINANCE,323
7,SPORTS,319
8,PHOTOGRAPHY,317
9,PERSONALIZATION,314


In [ ]:
fig = px.bar(
    category_counts.head(10),
    x="Category",
    y="Count",
    text="Count",
    title="Top 10 App Categories by Number of Apps"
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Number of Apps"
)

fig.show()

**Insight 1:** Some app categories have many more apps than others. The `FAMILY` category has the largest number of apps, followed by `GAME` and `TOOLS`. This shows that the Play Store is not balanced across categories.

In [ ]:
category_installs = (
    apps_story.groupby("Category", as_index=False)["Installs"]
    .sum()
    .sort_values("Installs", ascending=False)
)

category_installs.head(10)

,Category,Installs
14,GAME,35085862717
6,COMMUNICATION,32647241530
25,PRODUCTIVITY,14176070180
27,SOCIAL,14069841475
29,TOOLS,11450724500
11,FAMILY,10257701590
24,PHOTOGRAPHY,10088243130
21,NEWS_AND_MAGAZINES,7496210650
30,TRAVEL_AND_LOCAL,6868859300
31,VIDEO_PLAYERS,6221897200


In [ ]:
fig = px.bar(
    category_installs.head(10),
    x="Category",
    y="Installs",
    text="Installs",
    title="Top 10 Categories by Total Installs"
)

fig.update_layout(
    xaxis_title="Category",
    yaxis_title="Total Installs"
)

fig.show()

**Insight 2:** The categories with the most apps are not always the most popular. In this chart, `GAME` and `COMMUNICATION` have the highest total installs, even though `FAMILY` had the most apps in the previous chart. This means that more apps in a category does not always mean more users.

In [ ]:
fig = px.scatter(
    apps_story,
    x="Rating",
    y="Installs",
    color="Type",
    hover_data=["App", "Category"],
    title="Rating vs Installs",
    log_y=True,
    opacity=0.6
)

fig.update_layout(
    xaxis_title="Rating",
    yaxis_title="Installs (log scale)"
)

fig.show()

**Insight 3:** Apps with higher ratings do not always have more installs. Most highly installed apps have ratings around 4 or higher, but many apps with good ratings still have low installs. This suggests that rating is important, but it is not the only reason why an app becomes popular.

In [ ]:
type_counts = apps_story["Type"].value_counts().reset_index()
type_counts.columns = ["Type", "Count"]

fig = px.pie(
    type_counts,
    names="Type",
    values="Count",
    title="Distribution of Free vs Paid Apps"
)

fig.show()

**Insight 4:** Free apps dominate the Play Store. About 93.1% of apps are free, while only 6.9% are paid. This shows that most apps are offered without upfront payment.

## Summary of Main Insights

From these visualizations, four main patterns can be seen:

1. A few categories contain most of the apps, especially `FAMILY`, `GAME`, and `TOOLS`.
2. The most popular categories by installs are different from the categories with the most apps.
3. Higher ratings do not always mean higher installs.
4. Free apps are much more common than paid apps.

These results help define the main story of the dataset and give a clear direction for the final dashboard.